# 과제 1: 알라딘 베스트셀러 정적 크롤링 📚

2장에서 배운 requests + BeautifulSoup으로 알라딘 베스트셀러의 **제목, 링크, 할인가, 별점**을 1~3페이지에서 모아 CSV로 저장합니다.

| Step | 할 일 |
|---|---|
| 1 | 라이브러리 준비 |
| 2 | HTML 받아오기 |
| 3 | 책 **한 권** 덩어리 찾기 |
| 4 | 한 권에서 제목, 링크, 할인가, 별점 뽑기 |
| 5 | 한 페이지의 책 **전부** 뽑기 |
| 6 | **여러 페이지** 뽑기 |
| 7 | DataFrame으로 정리하고 CSV 저장 |

> ✏️ `____` 빈칸을 채우세요. 셀렉터는 **크롬 개발자 도구**(1장)로 직접 찾아야 합니다!

## Step 1. 라이브러리 준비

In [7]:
!pip install requests beautifulsoup4 pandas

import time

import requests
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://www.aladin.co.kr/shop/common/wbest.aspx"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0 Safari/537.36"}

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached charset_normalizer-3.5.2-cp312-cp312-win_amd64.whl.metadata (47 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached soupsieve-2.10-py3-none-any.whl.metadata (4.4 kB)
  Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp312-cp312-win_amd64.whl (207 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached pandas-3.0

## Step 2. HTML 받아오기

베스트셀러 주소 `...wbest.aspx?BestType=Bestseller&BranchType=1&...` 에서 `?` 뒤의 옵션들을 `params` 딕셔너리로 정리했어요.

💡 2장 복습: `requests.get(주소, params=..., headers=..., timeout=10)`

In [8]:
params = {"BestType": "Bestseller", "BranchType": 1, "CID": 0, "page": 1, "cnt": 1000, "SortOrder": 1}

response = requests.get(URL, params=params, headers=HEADERS, timeout=10)
print(response.status_code)   # 200이면 성공

soup = BeautifulSoup(response.text, 'html.parser')

200


## Step 3. 책 한 권 덩어리 찾기

개발자 도구로 첫 번째 책을 찍어보고, **책 한 권 전체를 감싸는 상자**를 찾으세요.

💡 힌트: 책마다 **똑같은 class 이름으로 반복되는** 상자가 있어요. 2장 지식인의 `.basic1 > li > dl`처럼요.

In [9]:
book = soup.select_one(".ss_book_box")
print(book.prettify()[:2000])

<div class="ss_book_box" itemid="403216369">
 <table border="0" cellpadding="0" cellspacing="0" width="100%">
  <tr>
   <td align="left" style="padding-right:5px;" valign="top" width="25">
    <table border="0" cellpadding="0" cellspacing="0" width="18">
     <tr>
      <td>
       <div style="text-align: center;">
        1.
       </div>
      </td>
     </tr>
     <tr>
      <td>
       <div style="text-align: center;">
        <input name="chkCart.K242131684" type="checkbox"/>
       </div>
      </td>
     </tr>
    </table>
   </td>
   <td align="left" valign="top" width="170">
    <table border="0" cellpadding="0" cellspacing="0" width="150">
     <tr>
      <td style="">
       <div class="cover_area">
        <a href="https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=403216369">
         <div class="flipcover_out" style="height:216px;">
          <div class="flipcover_in lcover_none">
           <img class="front_cover i_cover" src="https://image.aladin.co.kr/product/40321/63

## Step 4. 한 권에서 제목, 링크, 할인가, 별점 뽑기

- 제목과 링크는 같은 `a` 태그에 있어요: 글자는 `.text`, 주소는 `["href"]`
- 할인가, 별점도 개발자 도구로 class를 찾아보세요

In [11]:
title_tag = book.select_one(".bo3")
title = title_tag.text.strip()
link = title_tag["href"]
price = book.select_one(".ss_p2").text.strip()
rating = book.select_one(".star_score").text.strip()

print(title, link, price, rating, sep=" | ")

가호 | https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=403216369 | 16,650원 | 8.9


## Step 5. 한 페이지의 책 전부 뽑기

`select`로 책 덩어리를 **전부** 찾고, for문으로 Step 4를 반복합니다.

⚠️ **예외 처리**: 신간은 아직 별점이 없는 등, 모든 책의 정보가 똑같지 않아요. 정보가 빠진 책을 만나면 `select_one`이 `None`을 돌려주고, 거기에 `.text`를 붙이면 `AttributeError`가 납니다. `try-except`로 그런 책은 건너뛰세요.

In [12]:
books = soup.select(".ss_book_box")
print(len(books), "권 찾음")

rows = []
for book in books:
    try:
        title_tag = book.select_one(".bo3")
        rows.append({
            "title": title_tag.text.strip(),
            "link": title_tag["href"],
            "price": book.select_one(".ss_p2").text.strip(),
            "rating": book.select_one(".star_score").text.strip(),
        })
    except AttributeError:
        continue   # 정보가 빠진 책은 건너뛰기

print(len(rows), "권 저장")
rows[:3]

50 권 찾음
44 권 저장


[{'title': '가호',
  'link': 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=403216369',
  'price': '16,650원',
  'rating': '8.9'},
 {'title': '트렌드 코리아 2027',
  'link': 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=401344136',
  'price': '19,800원',
  'rating': '9.7'},
 {'title': '박완서의 낱말들',
  'link': 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=401813692',
  'price': '17,820원',
  'rating': '9.9'}]

## Step 6. 여러 페이지 뽑기

`params["page"]` 숫자만 바꿔가며 1~3페이지를 반복합니다. 요청 사이에는 `time.sleep()`으로 꼭 쉬어주세요!

In [13]:
rows = []
for page in range(1, 4):
    params["page"] = page
    response = requests.get(URL, params=params, headers=HEADERS, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    for book in soup.select(".ss_book_box"):
        try:
            title_tag = book.select_one(".bo3")
            rows.append({
                "title": title_tag.text.strip(),
                "link": title_tag["href"],
                "price": book.select_one(".ss_p2").text.strip(),
                "rating": book.select_one(".star_score").text.strip(),
            })
        except AttributeError:
            continue

    print(f"{page}페이지 완료, 지금까지 {len(rows)}권")
    time.sleep(1)

1페이지 완료, 지금까지 44권
2페이지 완료, 지금까지 90권
3페이지 완료, 지금까지 134권


## Step 7. DataFrame으로 정리하고 CSV 저장

💡 윈도우 엑셀에서 한글이 안 깨지게 하는 인코딩, 2장에서 배웠죠?

In [14]:
df = pd.DataFrame(rows)
df.to_csv("aladin.csv", index=False, encoding="utf-8-sig")
df.head()

,title,link,price,rating
0,가호,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"16,650원",8.9
1,트렌드 코리아 2027,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"19,800원",9.7
2,박완서의 낱말들,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"17,820원",9.9
3,싯다르타,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"7,200원",9.1
4,"세네카, 오늘을 빼앗기고 있는 당신에게",https://www.aladin.co.kr/shop/wproduct.aspx?It...,"16,200원",8.3


## 제출 전: .py로 변환하기

코드 리뷰를 위해 노트북을 `.py` 파일로 변환해서 제출합니다. 터미널에서 이 노트북이 있는 폴더로 이동한 뒤:

```bash
python -m jupyter nbconvert --to script hw1_aladin.ipynb
```